In [ ]:
import socket
import sqlite3
import json
import os
import threading
from datetime import datetime

DATABASE_NAME = 'cinema.db'
HOST = '127.0.0.1'
PORT = 65432

def init_db():
    conn = None
    try:
        conn = sqlite3.connect(DATABASE_NAME)
        cursor = conn.cursor()

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS movies (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL UNIQUE,
                cinema_room INTEGER NOT NULL CHECK(cinema_room >= 1 AND cinema_room <= 7),
                release_date TEXT NOT NULL,
                end_date TEXT NOT NULL,
                tickets_available INTEGER NOT NULL CHECK(tickets_available >= 0),
                ticket_price REAL NOT NULL CHECK(ticket_price > 0)
            )
        ''')

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS sales (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                movie_id INTEGER NOT NULL,
                customer_name TEXT NOT NULL,
                number_of_tickets INTEGER NOT NULL CHECK(number_of_tickets > 0),
                total REAL NOT NULL CHECK(total >= 0),
                sale_date TEXT NOT NULL,
                FOREIGN KEY (movie_id) REFERENCES movies(id) ON DELETE CASCADE
            )
        ''')

        initial_movies = [
            ("Sinners", 1, "2024-01-15", "2024-03-15", 100, 120.00),
            ("Corrina Corrina", 2, "2024-02-01", "2024-04-01", 150, 100.00),
            ("Lilo & Stitch", 3, "2024-03-10", "2024-05-10", 200, 90.00),
            ("Really Love", 4, "2024-04-05", "2024-06-05", 120, 110.00),
            ("One of Them Days", 5, "2024-05-20", "2024-07-20", 80, 130.00),
            ("Annie", 6, "2024-06-01", "2024-08-01", 180, 95.00),
            ("Moana", 7, "2024-07-10", "2024-09-10", 220, 85.00),
        ]

        cursor.execute("SELECT COUNT(*) FROM movies")
        if cursor.fetchone()[0] == 0:
            cursor.executemany('''
                INSERT INTO movies (title, cinema_room, release_date, end_date, tickets_available, ticket_price)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', initial_movies)
            print("Database populated with initial movie data.")
        else:
            print("Movies table already populated.")

        conn.commit()
        print("Database initialized successfully.")

    except sqlite3.Error as e:
        print(f"Database error during initialization: {e}")
        if conn:
            conn.rollback()
    finally:
        if conn:
            conn.close()

def db_operation(func):
    def wrapper(*args, **kwargs):
        conn = None
        try:
            conn = sqlite3.connect(DATABASE_NAME)
            conn.row_factory = sqlite3.Row
            cursor = conn.cursor()
            result = func(cursor, *args, **kwargs)
            conn.commit()
            return result
        except sqlite3.Error as e:
            if conn:
                conn.rollback()
            raise Exception(f"Database operation failed: {e}")
        finally:
            if conn:
                conn.close()
    return wrapper

@db_operation
def get_movies_from_db(cursor):
    cursor.execute("SELECT * FROM movies ORDER BY title")
    movies = [dict(row) for row in cursor.fetchall()]
    return movies

@db_operation
def add_movie_to_db(cursor, title, cinema_room, release_date, end_date, tickets_available, ticket_price):
    try:
        datetime.strptime(release_date, '%Y-%m-%d')
        datetime.strptime(end_date, '%Y-%m-%d')
    except ValueError:
        raise Exception("Invalid date format. Use YYYY-MM-DD.")

    if not (1 <= cinema_room <= 7):
        raise Exception("Cinema room must be between 1 and 7.")
    if not (tickets_available >= 0):
        raise Exception("Tickets available cannot be negative.")
    if not (ticket_price > 0):
        raise Exception("Ticket price must be positive.")
    if not title.strip():
        raise Exception("Movie title cannot be empty.")

    try:
        cursor.execute('''
            INSERT INTO movies (title, cinema_room, release_date, end_date, tickets_available, ticket_price)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (title, cinema_room, release_date, end_date, tickets_available, ticket_price))
        return {"id": cursor.lastrowid, "title": title}
    except sqlite3.IntegrityError as e:
        if "UNIQUE constraint failed: movies.title" in str(e):
            raise Exception(f"Movie with title '{title}' already exists.")
        else:
            raise Exception(f"Failed to add movie: {e}")

@db_operation
def update_movie_in_db(cursor, movie_id, title, cinema_room, release_date, end_date, tickets_available, ticket_price):
    try:
        datetime.strptime(release_date, '%Y-%m-%d')
        datetime.strptime(end_date, '%Y-%m-%d')
    except ValueError:
        raise Exception("Invalid date format. Use YYYY-MM-DD.")

    if not (1 <= cinema_room <= 7):
        raise Exception("Cinema room must be between 1 and 7.")
    if not (tickets_available >= 0):
        raise Exception("Tickets available cannot be negative.")
    if not (ticket_price > 0):
        raise Exception("Ticket price must be positive.")
    if not title.strip():
        raise Exception("Movie title cannot be empty.")

    cursor.execute("SELECT id FROM movies WHERE id = ?", (movie_id,))
    if cursor.fetchone() is None:
        raise Exception(f"Movie with ID {movie_id} not found.")

    try:
        cursor.execute('''
            UPDATE movies
            SET title = ?, cinema_room = ?, release_date = ?, end_date = ?, tickets_available = ?, ticket_price = ?
            WHERE id = ?
        ''', (title, cinema_room, release_date, end_date, tickets_available, ticket_price, movie_id))
        return {"id": movie_id, "title": title}
    except sqlite3.IntegrityError as e:
        if "UNIQUE constraint failed: movies.title" in str(e):
            raise Exception(f"Another movie with title '{title}' already exists.")
        else:
            raise Exception(f"Failed to update movie: {e}")

@db_operation
def delete_movie_from_db(cursor, movie_id):
    cursor.execute("SELECT id FROM movies WHERE id = ?", (movie_id,))
    if cursor.fetchone() is None:
        raise Exception(f"Movie with ID {movie_id} not found.")

    cursor.execute("DELETE FROM movies WHERE id = ?", (movie_id,))
    return {"id": movie_id}

@db_operation
def record_ticket_sale_in_db(cursor, movie_id, customer_name, number_of_tickets):
    if number_of_tickets <= 0:
        raise Exception("Number of tickets must be positive.")
    if not customer_name.strip():
        raise Exception("Customer name cannot be empty.")

    cursor.execute("SELECT tickets_available, ticket_price, title FROM movies WHERE id = ?", (movie_id,))
    movie_info = cursor.fetchone()

    if movie_info is None:
        raise Exception(f"Movie with ID {movie_id} not found.")

    tickets_available = movie_info['tickets_available']
    ticket_price = movie_info['ticket_price']
    movie_title = movie_info['title']

    if tickets_available < number_of_tickets:
        raise Exception(f"Not enough tickets available for '{movie_title}'. Only {tickets_available} left.")

    total_cost = number_of_tickets * ticket_price
    sale_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    cursor.execute('''
        UPDATE movies
        SET tickets_available = tickets_available - ?
        WHERE id = ?
    ''', (number_of_tickets, movie_id))

    cursor.execute('''
        INSERT INTO sales (movie_id, customer_name, number_of_tickets, total, sale_date)
        VALUES (?, ?, ?, ?, ?)
    ''', (movie_id, customer_name, number_of_tickets, total_cost, sale_date))

    return {
        "movie_id": movie_id,
        "movie_title": movie_title,
        "customer_name": customer_name,
        "number_of_tickets": number_of_tickets,
        "total_cost": total_cost,
        "tickets_remaining": tickets_available - number_of_tickets
    }

def handle_client(conn, addr):
    print(f"Connected by {addr}")
    try:
        while True:
            data = conn.recv(4096)
            if not data:
                break

            try:
                request = json.loads(data.decode('utf-8'))
                action = request.get('action')
                payload = request.get('data', {})
                print(f"Received request: Action='{action}', Payload={payload}")

                response_data = {}
                status = "success"
                message = "Operation successful."

                if action == 'get_movies':
                    response_data = get_movies_from_db()
                elif action == 'add_movie':
                    response_data = add_movie_to_db(
                        payload.get('title'),
                        payload.get('cinema_room'),
                        payload.get('release_date'),
                        payload.get('end_date'),
                        payload.get('tickets_available'),
                        payload.get('ticket_price')
                    )
                elif action == 'update_movie':
                    response_data = update_movie_in_db(
                        payload.get('id'),
                        payload.get('title'),
                        payload.get('cinema_room'),
                        payload.get('release_date'),
                        payload.get('end_date'),
                        payload.get('tickets_available'),
                        payload.get('ticket_price')
                    )
                elif action == 'delete_movie':
                    response_data = delete_movie_from_db(payload.get('id'))
                elif action == 'record_sale':
                    response_data = record_ticket_sale_in_db(
                        payload.get('movie_id'),
                        payload.get('customer_name'),
                        payload.get('number_of_tickets')
                    )
                else:
                    status = "error"
                    message = "Unknown action."

            except json.JSONDecodeError:
                status = "error"
                message = "Invalid JSON format in request."
            except Exception as e:
                status = "error"
                message = str(e)
                print(f"Server error during request processing: {e}")

            response = {"status": status, "message": message, "data": response_data}
            conn.sendall(json.dumps(response).encode('utf-8'))

    except ConnectionResetError:
        print(f"Client {addr} disconnected unexpectedly.")
    except Exception as e:
        print(f"Error handling client {addr}: {e}")
    finally:
        conn.close()
        print(f"Connection with {addr} closed.")

def start_server():
    init_db()

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORT))
        s.listen()
        print(f"Server listening on {HOST}:{PORT}")

        while True:
            conn, addr = s.accept()
            handle_client(conn, addr)

if __name__ == "__main__":
    start_server()

Movies table already populated.
Database initialized successfully.
Server listening on 127.0.0.1:65432
Connected by ('127.0.0.1', 56852)
Received request: Action='get_movies', Payload={}
